# Data Overview
Reveals basic features about the data such as label distribution, author distribution, inter-annotator reliability, etc.

In [1]:
import pandas as pd

## Section 1: Main Data Overview

In [2]:
df = pd.read_csv("data.csv")

entries = df.shape[0]

print(f"{entries} entries")
print(f"{len(set(df["author"]))} unique post authors")
print(f"{df[~df["annotators"].str.contains("human")].shape[0]} human-reviewed AI annotations")
print("\nLabel distribution:")
vc = df["label"].value_counts()

for label, count in vc.items():
    percent = count * 100 / entries
    print(f"{label:<20} {count:<4} {percent:.2f}%")

213 entries
169 unique post authors
25 human-reviewed AI annotations

Label distribution:
artistic_critique    89   41.78%
external_narrative   84   39.44%
fandom_expression    40   18.78%


## Section 2: Inter-Annotator Reliability

In [36]:
h2_df = pd.read_csv("human2_annot.csv")

# Rename label columns in each dataframe
df_h1 = df[["id", "text", "label"]].rename(columns={"label": "label_h1"})
h2_h2 = h2_df[["id", "label"]].rename(columns={"label": "label_h2"})

# Merge on id (inner join keeps only matching ids - like zip())
merged_df = df_h1.merge(h2_h2, on="id", how="inner").dropna().reset_index(drop=True)

# Truncate text
merged_df["text"] = merged_df["text"].apply(
    lambda x: x[:100] + "..." if len(x) > 100 else x
)

shape = merged_df.shape
agreement = merged_df["label_h1"] == merged_df["label_h2"]
n_match = int(agreement.sum())
pct_match = n_match * 100 / shape[0]

print(f"Merged dataframe shape: {shape}")
print(f"Agreement: {n_match}/{shape[0]} ({pct_match:.2f}%)")

disagreement_df = merged_df[~agreement]
print(f"Sample disagreement IDs: {disagreement_df.sample(n=5,random_state=42)["id"].to_list()}")
print("All disagreements:")
disagreement_df


Merged dataframe shape: (39, 4)
Agreement: 27/39 (69.23%)
Sample disagreement IDs: [35, 31, 3, 27, 16]
All disagreements:


,id,text,label_h1,label_h2
2,3,"Even upon first listen, ""Getaway Car"" became m...",artistic_critique,fandom_expression
4,5,"Haven't heard it yet, but the Taylor sub is we...",external_narrative,fandom_expression
5,6,This is the first time I've been witness to a ...,external_narrative,fandom_expression
7,8,I had a dream last night where I heard Dress a...,fandom_expression,external_narrative
9,10,The old Popheads can’t come to the phone right...,fandom_expression,external_narrative
15,16,Mark My Words. End Game will be an 8-week #1 B...,external_narrative,fandom_expression
20,22,This is... not good. I'm going back to 1989.,artistic_critique,fandom_expression
21,23,Only heard the album once so far but I'm surpr...,artistic_critique,fandom_expression
25,27,Wow this album is not good. I was hoping for s...,artistic_critique,fandom_expression
29,31,New Years Day is going straight into my Christ...,external_narrative,fandom_expression
